# Evaluate and run checks (smoke)

Purpose
-------

- Optionally run a short smoke training run and collect the best checkpoint;
- Run batched inference using the checkpoint and the prepared manifest;
- Produce simple metrics and a loss plot (if logs available);
- Save an evaluation report and generated WAVs for manual inspection.

Safety
------

- Training is disabled by default (dry-run). Enable explicitly and monitor resources.

In [ ]:
import json
import shutil
import subprocess
import time
from pathlib import Path
from statistics import mean
from typing import Optional

import matplotlib.pyplot as plt
import soundfile as sf
from TTS.api import TTS

project_root = Path(".")
manifest_path = project_root / "data/processed/manifests/smoke_manifest.jsonl"
config_path = project_root / "configs/smoke_vits.yaml"
run_script_path = project_root / "scripts/run_finetune_smoke.bat"
outputs_dir = project_root / "outputs/smoke_vits"
eval_dir = outputs_dir / "eval"
eval_dir.mkdir(parents=True, exist_ok=True)

run_training = False
training_timeout_seconds = 600
tail_seconds = 20
num_eval_samples = 10

## Confirm required artifacts exist

- Ensures the manifest, config and helper script were produced by Notebook 06;
- Fails early if something is missing

In [ ]:
missing = []

for item in (manifest_path, config_path, run_script_path):
    if not item.exists():
        missing.append(str(item))

if missing:
    print("Missing required artifacts (run notebook 06 first):")
    for missing_item in missing:
        print(" -", missing_item)

else:
    print("All artifacts present:")
    print(" - manifest:", manifest_path)
    print(" - config:", config_path)
    print(" - run script:", run_script_path)

## Optionally launch shor training helper (dry-run by default)

- If `run_training = True` this will run the `.bat` helper from Notebook 06 and tail logs for short time;
- This is intended only for tiny smoke runs, prefer manual CLI if you want long training.

In [ ]:
def run_and_tail(
    script_path: Path, timeout: int = 600, tail_seconds: int = 20
) -> Optional[int]:
    if not script_path.exists():
        raise FileExistsError(f"Run script not found: {script_path}")

    log_path = outputs_dir / "training_run.log"

    with open(log_path, "wb") as log_file:
        proc = subprocess.Popen(
            str(script_path), shell=True, stdout=log_file, stderr=subprocess.STDOUT
        )

    print(f"Launched helper PID={proc.pid}, logging to {log_path}")

    try:
        time.sleep(min(tail_seconds, timeout))
        if log_path.exists():
            print("---- last 30 lines of log ----")
            print(
                "".join(
                    log_path.read_text(encoding="utf-8", errors="ignore").splitlines()[
                        -30:
                    ]
                )
            )

        proc.wait(timeout=timeout - tail_seconds)
    except subprocess.TimeoutExpired:
        proc.kill()
        print("Training helper killed due to timeout")
        return None

    return proc.returncode


if run_training:
    run_return = run_and_tail(
        run_script_path, timeout=training_timeout_seconds, tail_seconds=tail_seconds
    )
    print("Helper return code:", run_return)
else:
    print("run_training is False (dry run). No training launched.")

## Find best checkpoint

- Heuristic: pick latest checkpoint by modification time from outputs directory (adjust if you have validation-based naming);
- Copy the selected checkpoint to `eval_dir` for safe inference.

In [ ]:
def find_latest_checkpoint(outputs: Path) -> Optional[Path]:
    checkpoint_patterns = ["**/*.pth.tar", "**/*.ckpt", "**/*.pt", "**/best_model.pth"]
    candidates = []
    for pattern in checkpoint_patterns:
        candidates.extend(outputs.glob(pattern))

    if not candidates:
        return None

    latest = max(candidates, key=lambda path: path.stat().st_mtime)
    return latest


latest_checkpoint = find_latest_checkpoint(outputs_dir)
if latest_checkpoint:
    target_checkpoint = eval_dir / latest_checkpoint.name
    shutil.copy2(latest_checkpoint, target_checkpoint)
    print("Copied latest checkpount to eval dir:", target_checkpoint)
else:
    print("No checkpoint found in outputs, ensure a training run produced checkpoints.")

## Batch inference using the selected checkpoint (or a prebuilt model)

- Reads up to `num_eval_samples` records from the manifest, synthesizes audio and writes WAVs to `eval_dir/wavs/`;
- Uses a prebuilt model if no checkpoint available. 

In [ ]:
eval_wavs_dir = eval_dir / "wave"
eval_wavs_dir.mkdir(parents=True, exist_ok=True)


def load_manifest_examples(manifest: Path, max_examples: int = 10):
    entries = []
    if not manifest.exists():
        return entries

    with open(manifest, "r", encoding="utf-8") as file:
        for idx, line in enumerate(file):
            if idx >= max_examples:
                break

            try:
                rec = json.loads(line)
                entries.append(rec)
            except Exception:
                continue

    return entries


examples = load_manifest_examples(manifest_path, num_eval_samples)

if not examples:
    print('No manifest entries found for inference, check "manifest_path".')
else:
    model_to_use = None
    if latest_checkpoint and (eval_dir / latest_checkpoint.name).exists():
        model_to_use = str((eval_dir / latest_checkpoint).resolve())
        print("Using local checkpoint for inference:", model_to_use)
    else:
        model_to_use = "tts_models/pt/cv/vits"
        print("No local checkpoint - using prebuilt model:", model_to_use)

    tts = TTS(model_name=model_to_use)
    sample_rate = getattr(
        getattr(tts, "synthesizer", None), "output_sample_rate", 22050
    )

    written = []

    for idx, rec in enumerate(examples, start=1):
        text = rec.get("text") or rec.get("transcript") or rec.get("sentence") or ""

        if not text:
            continue

        wav = tts.tts(text)
        output_path = eval_wavs_dir / f"eval_{idx}.wav"
        sf.write(str(output_path), wav, sample_rate)
        written.append(output_path)
        print("Wrote:", output_path)

    print(f"Batch inference finished. Wrote {len(written)} files to {eval_wavs_dir}")

## Compute simple metrics and plot training loss (if available)

- Metrics: number of eval WAVs, mean duration, manifest vs synthesized count;
- If a training log (CSV/JSON) exists, plot loss curve and save figure.

In [ ]:
generated_files = list((eval_dir / "wavs").glob("*.wav"))
durations = []

for file in generated_files:
    try:
        info = sf.info(str(file))
        durations.append(info.frames / info.samplerate)
    except Exception:
        continue

print("Generated WAV count:", len(generated_files))
if durations:
    print(
        f"Duration stats: count={len(durations)}, mena={round(mean(durations)),3}, min={min(durations)}, max={max(durations)}"
    )

log_csv = outputs_dir / "training_log.csv"
if log_csv.exists():
    data = {"step": [], "loss": []}
    with open(log_csv, "r", encoding="utf-8") as file:
        headers = file.readline().strip().split(",")
        for line in file:
            vals = line.strip().split(",")
            row = dict(zip(headers, vals))
            try:
                data["step"].append(int(row.get("step", 0)))
                data["loss"].append(int(row.get("loss", 0.0)))
            except Exception:
                continue

    if data["step"]:
        plt.figure()
        plt.plot(data["step"], data["loss"], "-o")
        plt.xlabel("step")
        plt.ylabel("loss")
        plt.title("Training loss")
        fig_path = eval_dir / "loss_plot.png"
        plt.savefig(fig_path)
        print("Saved loss plot to", fig_path)

else:
    print("No training_log.csv found, skip loss plotting.")

## Report & next steps

- The notebook saved generated WAVs in outputs/smoke_vits/eval/wavs and a copy of the checkpoint in outputs/smoke_vits/eval.
- Recommended: listen to generated files, inspect the loss plot if available, and, if acceptable, run a longer fine-tune with adjusted config.